In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import splink
from splink.duckdb.linker import DuckDBLinker
import pandas as pd

os.chdir(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraa')

df_l = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\df_l_censo_sin_snsa.parquet')
df_r = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\df_r_rraa_sin_snsa.parquet')

#rraa = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\rraa.parquet')
censo = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\censo.parquet')
extranjeros_vinculados = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\extranjeros_vinculados.parquet')

linker = DuckDBLinker([df_l, df_r])
linker.load_model(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\modelos\modelo4.json')

### actualizo la vinculación para las tablas de censo a la versión del 15-10

In [2]:
from load_data.censo import get_personas

censo_15_10 = get_personas(["perna02_r", "perna02_r_imp"])

df_l2 = df_l[(df_l["unique_id"].isin(censo_15_10["id_censo"]))].merge(censo_15_10, left_on="unique_id", right_on="id_censo")

df_l2["fecha_nacimiento"] = pd.to_datetime(np.where(
    df_l2.apply(lambda row: row["perna02_r_imp"] == 1, axis = 1), 
    pd.NaT, 
    df_l2["perna02_r"]), 
    errors="coerce")

df_l2["ano_nacimiento"] = df_l2["fecha_nacimiento"].dt.year
df_l2["mes_nacimiento"] = df_l2["fecha_nacimiento"].dt.month
df_l2["dia_nacimiento"] = df_l2["fecha_nacimiento"].dt.day

df_l2["fecha_nacimiento"] = df_l2["fecha_nacimiento"].astype(str)

In [3]:
extranjeros_vinculados2 = extranjeros_vinculados[extranjeros_vinculados["unique_id_l"].astype(float).isin(censo_15_10["id_censo"])]

In [5]:
df_l = df_l2.drop(columns=["perna02_r_imp", "perna02_r", "id_censo"])
extranjeros_vinculados = extranjeros_vinculados2

linker = DuckDBLinker([df_l, df_r])
linker.load_model(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\modelos\modelo4.json')

In [ ]:
# df_predictions = linker.predict(threshold_match_probability=0.2)

# predictions = df_predictions.as_pandas_dataframe()

# max_match_probabilities = predictions.groupby('unique_id_l')['match_weight'].idxmax()

# predictions_max_match_prob = predictions.loc[max_match_probabilities]

# predictions_max_match_prob = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\pred_modelo4.parquet')

### .

In [2]:
from utils.saveFile import guardar_como_parquet

#guardar_como_parquet(predictions_max_match_prob, r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\pred_modelo4_23_10.parquet')

predictions_max_match_prob = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\pred_modelo4.parquet')

In [3]:
cortes = [0.2, 0.9, 0.999, 1]
umbrales = [3, 2, 1]
umbrales_descripcion = ["match no confiable (0.2 - 0.9)", "match poco confiable (0.9 - 0.999)", "match confiable (> 0.999)"]

predictions_max_match_prob["id_umbral_vinculo"] = pd.cut(
    predictions_max_match_prob["match_probability"],
    bins = cortes,
    labels = umbrales
)

predictions_max_match_prob["umbral_vinculo"] = pd.cut(
    predictions_max_match_prob["match_probability"],
    bins = cortes,
    labels = umbrales_descripcion
)

predictions_max_match_prob["coincidencia_documento"] = np.where(
    predictions_max_match_prob["gamma_documento"] == 3,
    1,
    0
)

documentos = df_l.rename(columns={"documento":"documento_l"})["documento_l"].value_counts()

predictions_max_match_prob["doc_unico"] = predictions_max_match_prob["documento_l"].apply(lambda x: 1 if pd.notna(x) and documentos.get(x, 0) == 1 else 0)

vinculados = predictions_max_match_prob[["unique_id_l", "unique_id_r", "match_probability", "id_umbral_vinculo", "umbral_vinculo", "coincidencia_documento", "doc_unico", "gamma_documento", "gamma_fecha_nacimiento", "gamma_primer_nombre", "gamma_primer_apellido"]].rename(columns = {"unique_id_l":"id_censo", "unique_id_r":"id_estadistico_persona"})

In [4]:
condicion = ((vinculados["id_umbral_vinculo"] == 1) | ((vinculados["id_umbral_vinculo"].isin([2])) & (((vinculados["gamma_documento"] == 3) & (vinculados["doc_unico"] == 1)) | 
                         ((vinculados["gamma_fecha_nacimiento"] == 3) & (vinculados["gamma_primer_nombre"].isin([1,2])) & (vinculados["gamma_primer_apellido"].isin([1,2]))))))

In [5]:
vinculados.loc[condicion, "es_match"] = 1

In [6]:
from load_data.censo import get_personas

censo = get_personas(['caso_r', 'vivid_r', 'hogid_r', 'departamento_r', 'localidad_r', 'zona_r', 'seccion_r', 'segmento_r', 'municipio', 'direccionid_r', 'perph02_r', 'perna01_r', 'fuente', 'tipo_caso', 'perna02_r', 'perna02_r_imp'])

censo["fecha_nacimiento"] = pd.to_datetime(censo["perna02_r"], errors = "coerce")

#censo.loc[censo["perna02_r_imp"] == 1, "fecha_nacimiento"] = pd.to_datetime(censo["perna02_r"], errors = "coerce")

In [8]:
censo = censo.rename(columns={"perph02_r":"id_sexo", "departamento_r":"id_departamento", "localidad_r":"id_localidad", "perna01_r":"edad"})

In [9]:
extranjeros_vinculados["id_estadistico_persona_ext"] = extranjeros_vinculados["unique_id_r"].str.split('_').str[0]

extranjeros_vinculados = extranjeros_vinculados.rename(columns={"unique_id_l":"id_censo"})
extranjeros_vinculados["id_censo"] = extranjeros_vinculados["id_censo"].astype("int64")

In [10]:
tabla_de_vinculos = censo[["id_censo", "caso_r", "vivid_r", "hogid_r", "tipo_caso", "fecha_nacimiento", "perna02_r_imp", "edad", "id_sexo", "id_departamento", "id_localidad", "zona_r", "seccion_r", "segmento_r", "direccionid_r", "municipio"]].merge(vinculados, how = "left").rename(columns={"perna02_r_imp":"fecha_nacimiento_imputada"})

tabla_de_vinculos['id_umbral_vinculo'] = tabla_de_vinculos['id_umbral_vinculo'].cat.add_categories([0,4])
tabla_de_vinculos['umbral_vinculo'] = tabla_de_vinculos['umbral_vinculo'].cat.add_categories(["no match", "match documento extranjero"])

tabla_de_vinculos.loc[(tabla_de_vinculos["id_censo"].isin(extranjeros_vinculados["id_censo"])), ["es_match", "id_umbral_vinculo", "umbral_vinculo", "coincidencia_documento"]] = [1, 4, "match documento extranjero", np.nan]
tabla_de_vinculos = tabla_de_vinculos.merge(extranjeros_vinculados, how = "left", on = "id_censo")
tabla_de_vinculos.loc[tabla_de_vinculos["id_umbral_vinculo"] == 4, "id_estadistico_persona"] = tabla_de_vinculos[tabla_de_vinculos["id_umbral_vinculo"] == 4]["id_estadistico_persona_ext"]

tabla_de_vinculos.loc[(pd.isnull(tabla_de_vinculos["id_estadistico_persona"])), ["id_umbral_vinculo", "umbral_vinculo", "coincidencia_documento"]] = [0, "no match", 0]
tabla_de_vinculos.loc[pd.isnull(tabla_de_vinculos["es_match"]), "es_match"] = 0

tabla_de_vinculos.loc[pd.isnull(tabla_de_vinculos["fecha_nacimiento"]), ["fecha_nacimiento_imputada"]] = 0
tabla_de_vinculos.loc[tabla_de_vinculos["id_sexo"] == 9898, "id_sexo"] = np.nan
tabla_de_vinculos.loc[tabla_de_vinculos["edad"] == 9898, "edad"] = np.nan

tabla_de_vinculos = tabla_de_vinculos.drop(columns=["unique_id_r", "id_estadistico_persona_ext"])

tabla_de_vinculos.loc[tabla_de_vinculos["id_localidad"]=='9898', "id_localidad"] = np.nan

tabla_de_vinculos["id_estadistico_persona"] = pd.to_numeric(tabla_de_vinculos["id_estadistico_persona"], errors = "coerce").astype("Int64")

C:\Users\lpescetto\AppData\Local\Temp\ipykernel_1504\3998509545.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['464142088888' '11295299' '334466388888' ... '200851888888'
 '488504988888' '103597099999']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tabla_de_vinculos.loc[tabla_de_vinculos["id_umbral_vinculo"] == 4, "id_estadistico_persona"] = tabla_de_vinculos[tabla_de_vinculos["id_umbral_vinculo"] == 4]["id_estadistico_persona_ext"]


In [11]:
tabla_de_vinculos.value_counts(["id_umbral_vinculo", "es_match"])

id_umbral_vinculo  es_match
1                  1.0         2971244
0                  0.0           77031
2                  1.0           46781
3                  0.0           25688
2                  0.0           13483
4                  1.0            4586
Name: count, dtype: int64

In [12]:
tabla_de_vinculos.value_counts("id_departamento", dropna = False)

id_departamento
01    1192782
03     544735
10     186209
05     122216
15     120464
11     108895
16     107682
13      93254
18      84100
04      79231
17      73593
14      69864
02      66890
08      64168
06      54394
09      53035
12      51304
19      42375
07      23622
Name: count, dtype: int64

In [13]:
a = tabla_de_vinculos.value_counts("id_localidad", dropna = False)

In [14]:
from utils.postgresFunctions import crear_tabla_en_batches

crear_tabla_en_batches(tabla_de_vinculos, "registros_prod", "censo", "vinculacion_censo_rraa")

Creando tabla censo.vinculacion_censo_rraa en DW
Progreso: 0.00%
Progreso: 25.00%
Progreso: 50.00%
Progreso: 75.00%
La tabla censo.vinculacion_censo_rraa fue actualizada.


In [15]:
from utils.postgresFunctions import crear_tabla_en_batches

crear_tabla_en_batches(tabla_de_vinculos, "registros_test", "censo", "vinculacion_censo_rraa")

Creando tabla censo.vinculacion_censo_rraa en DW
Progreso: 0.00%
Progreso: 25.00%
Progreso: 50.00%
Progreso: 75.00%
La tabla censo.vinculacion_censo_rraa fue actualizada.


In [5]:
from utils.db_connections.registros import make_dw_conection

regprod = make_dw_conection("registros_prod")

vinculacion_censo_rraa = pd.read_sql("select * from censo.vinculacion_censo_rraa",
                                     regprod)

In [ ]:
from utils.db_connections.registros import make_dw_conection

dw_connection = make_dw_conection("censo_prod")
dw_connection.close()